# Preparación de datos 

## Objetivo
Limpiar y preparar los datos de participación electoral para avanzar a la
**Fase 3 — Modelo de riesgo**.

## Contexto
Este notebook **continúa** de la Fase 1 (`exploracion_datos.ipynb`):
- Se cargan los mismos CSV del Conteo Censal de Participación 2024 del INE.
- Se aplican los criterios de limpieza y validación detectados en la exploración.
- Se derivan indicadores de abstención / participación con denominadores seguros.

## Fases asociadas
- Fase 1: Comprensión de la información
- Fase 2: Preparación de los datos (actual)

In [1]:
import glob
import os

import numpy as np
import pandas as pd

DATA_DIR = '../DatosAbiertos_DECEyEC_ConteosCensales2024'
ANIOS = [2024, 2021, 2018, 2015, 2012, 2009]

files = []
for anio in ANIOS:
    files += sorted(glob.glob(os.path.join(DATA_DIR, f'datosabiertos_deceyec_conteoscensales{anio}_*.csv')))

files = sorted(set(files))
print(len(files))

32


## 1. Carga de los datos

In [2]:
frames = []
for f in files:
    frames.append(pd.read_csv(f, encoding='utf-8-sig'))

df = pd.concat(frames, ignore_index=True)
print(df.shape)
df.head()

(9773730, 16)


,AELEC,FELECCION,EDOCVE,EDONOM,MPIOCVE,MPIONOM,SECCION,TIPOSEC,DEF,DEL,SEXO,EDAD,LN,SV,NV,NS
0,2024,06/02/2024,1,AGUASCALIENTES,5,Jesús María,413,M,1,NaN,0,21,133,85,48,0
1,2024,06/02/2024,1,AGUASCALIENTES,5,Jesús María,413,M,1,NaN,0,57,77,65,12,0
2,2024,06/02/2024,1,AGUASCALIENTES,7,Rincón de Romos,437,M,1,1.0,0,23,36,14,22,0
3,2024,06/02/2024,1,AGUASCALIENTES,7,Rincón de Romos,450,M,1,1.0,0,37,14,8,6,0
4,2024,06/02/2024,1,AGUASCALIENTES,7,Rincón de Romos,450,M,1,1.0,0,40,15,7,8,0


In [3]:
# Calidad: verificar que todos los años tengan el mismo esquema de columnas
columnas_por_anio = {}
for anio in ANIOS:
    for f in sorted(glob.glob(os.path.join(DATA_DIR, f'datosabiertos_deceyec_conteoscensales{anio}_*.csv'))):
        entidad = os.path.basename(f).split(f'conteoscensales{anio}_')[1].split('.')[0]
        columnas_por_anio[entidad] = list(pd.read_csv(f, encoding='utf-8-sig', nrows=0).columns)

esquemas = set(tuple(v) for v in columnas_por_anio.values())
print(f'Entidades con datos: {len(columnas_por_anio)}')
print(f'Esquemas únicos detectados: {len(esquemas)}')
for entidad, cols in columnas_por_anio.items():
    print(entidad, list(cols))

if len(esquemas) == 1:
    print('OK: todos los archivos comparten el mismo esquema')
else:
    print('⚠️ Esquemas diferentes entre años/entidades: requiere revisión')

Entidades con datos: 32
Esquemas únicos detectados: 1
ags ['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']
bc ['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']
bcs ['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']
camp ['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']
cdmx ['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']
chih ['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SECCION', 'TIPOSEC', 'DEF', 'DEL', 'SEXO', 'EDAD', 'LN', 'SV', 'NV', 'NS']
chis ['AELEC', 'FELECCION', 'EDOCVE', 'EDONOM', 'MPIOCVE', 'MPIONOM', 'SEC

## 2. Limpieza de variables

In [4]:
df_clean = df.copy()

# --- Normalización de tipos ---
df_clean['AELEC'] = pd.to_numeric(df_clean['AELEC'], errors='coerce').astype('Int64')
df_clean['FELECCION'] = pd.to_datetime(df_clean['FELECCION'], format='%d/%m/%Y', errors='coerce')
df_clean['EDOCVE'] = pd.to_numeric(df_clean['EDOCVE'], errors='coerce').astype('Int64')
df_clean['EDONOM'] = df_clean['EDONOM'].astype(str).str.strip().str.upper()
df_clean['MPIOCVE'] = pd.to_numeric(df_clean['MPIOCVE'], errors='coerce').astype('Int64')
df_clean['MPIONOM'] = df_clean['MPIONOM'].astype(str).str.strip()
df_clean['SECCION'] = pd.to_numeric(df_clean['SECCION'], errors='coerce').astype('Int64')
df_clean['TIPOSEC'] = df_clean['TIPOSEC'].astype(str).str.strip().str.upper()
df_clean['DEF'] = pd.to_numeric(df_clean['DEF'], errors='coerce').astype('Int64')
df_clean['DEL'] = pd.to_numeric(df_clean['DEL'], errors='coerce').astype('Int64')
df_clean['SEXO'] = pd.to_numeric(df_clean['SEXO'], errors='coerce').astype('Int64')
df_clean['EDAD'] = pd.to_numeric(df_clean['EDAD'], errors='coerce').astype('Int64')
df_clean['LN'] = pd.to_numeric(df_clean['LN'], errors='coerce').astype('Int64')
df_clean['SV'] = pd.to_numeric(df_clean['SV'], errors='coerce').astype('Int64')
df_clean['NV'] = pd.to_numeric(df_clean['NV'], errors='coerce').astype('Int64')
df_clean['NS'] = pd.to_numeric(df_clean['NS'], errors='coerce').astype('Int64')

print(df_clean.dtypes)
df_clean.head()

AELEC                 Int64
FELECCION    datetime64[us]
EDOCVE                Int64
EDONOM                  str
MPIOCVE               Int64
MPIONOM                 str
SECCION               Int64
TIPOSEC                 str
DEF                   Int64
DEL                   Int64
SEXO                  Int64
EDAD                  Int64
LN                    Int64
SV                    Int64
NV                    Int64
NS                    Int64
dtype: object


,AELEC,FELECCION,EDOCVE,EDONOM,MPIOCVE,MPIONOM,SECCION,TIPOSEC,DEF,DEL,SEXO,EDAD,LN,SV,NV,NS
0,2024,2024-02-06,1,AGUASCALIENTES,5,Jesús María,413,M,1,<NA>,0,21,133,85,48,0
1,2024,2024-02-06,1,AGUASCALIENTES,5,Jesús María,413,M,1,<NA>,0,57,77,65,12,0
2,2024,2024-02-06,1,AGUASCALIENTES,7,Rincón de Romos,437,M,1,1,0,23,36,14,22,0
3,2024,2024-02-06,1,AGUASCALIENTES,7,Rincón de Romos,450,M,1,1,0,37,14,8,6,0
4,2024,2024-02-06,1,AGUASCALIENTES,7,Rincón de Romos,450,M,1,1,0,40,15,7,8,0


## 3. Validaciones y registros problemáticos

In [5]:
# Valores nulos por columna
df_clean.isna().sum()

# Valores esperados: SEXO en {0, 1, 2}; EDAD >= 18; conteos >= 0
sexo_invalido = df_clean.loc[~df_clean['SEXO'].isin([0, 1, 2]), ['EDONOM', 'MPIOCVE', 'SECCION', 'TIPOSEC', 'SEXO']]
edad_invalida = df_clean.loc[df_clean['EDAD'] < 18, ['EDONOM', 'MPIOCVE', 'SECCION', 'TIPOSEC', 'EDAD']]
conteos_negativos = df_clean.loc[
    (df_clean['SV'] < 0) | (df_clean['NV'] < 0) | (df_clean['NS'] < 0) | (df_clean['LN'] < 0),
    ['EDONOM', 'MPIOCVE', 'SECCION', 'TIPOSEC', 'LN', 'SV', 'NV', 'NS'],
]

print(f'Filas con SEXO fuera de rango: {len(sexo_invalido)}')
print(f'Filas con EDAD < 18: {len(edad_invalida)}')
print(f'Filas con conteos negativos: {len(conteos_negativos)}')

Filas con SEXO fuera de rango: 0
Filas con EDAD < 18: 0
Filas con conteos negativos: 0


## 4. Indicadores derivados 

In [6]:
denominador = df_clean['SV'].fillna(0) + df_clean['NV'].fillna(0)
tasa_participacion = df_clean['SV'].fillna(0) / denominador.where(denominador > 0, 1)
tasa_abstencion = df_clean['NV'].fillna(0) / denominador.where(denominador > 0, 1)

df_clean['tasa_participacion'] = tasa_participacion
df_clean['tasa_abstencion'] = tasa_abstencion

print(df_clean[['SV', 'NV', 'NS', 'tasa_participacion', 'tasa_abstencion']].head())

   SV  NV  NS  tasa_participacion  tasa_abstencion
0  85  48   0            0.639098         0.360902
1  65  12   0            0.844156         0.155844
2  14  22   0            0.388889         0.611111
3   8   6   0            0.571429         0.428571
4   7   8   0            0.466667         0.533333


## 5. Indicadores por zona (entidad federativa / municipio)

In [7]:
resumen_entidad = (
    df_clean.groupby(['AELEC', 'EDONOM'])
    .agg(
        LN=('LN', 'sum'),
        SV=('SV', 'sum'),
        NV=('NV', 'sum'),
        NS=('NS', 'sum'),
    )
    .assign(
        tasa_abstencion=lambda x: x['NV'] / (x['SV'] + x['NV']).replace(0, np.nan),
        tasa_participacion=lambda x: x['SV'] / (x['SV'] + x['NV']).replace(0, np.nan),
    )
    .sort_values('tasa_abstencion', ascending=False)
)

resumen_entidad.head()

LN       SV       NV      NS  tasa_abstencion  \
AELEC EDONOM                                                                    
2024  BAJA CALIFORNIA      3197817  1396335  1571676  229806         0.529538   
      SONORA               2291013  1018161  1035322  237530         0.504179   
      CHIHUAHUA            3082363  1360070  1262547  459746         0.481407   
      BAJA CALIFORNIA SUR   613613   310713   274921   27979         0.469442   
      DURANGO              1397549   735959   633637   27953         0.462645   

                           tasa_participacion  
AELEC EDONOM                                   
2024  BAJA CALIFORNIA                0.470462  
      SONORA                         0.495821  
      CHIHUAHUA                      0.518593  
      BAJA CALIFORNIA SUR            0.530558  
      DURANGO                        0.537355